# Dog Breed Identification using Transfer Learning (VGG19)

This notebook provides a complete guide to building a Dog Breed Identification system. It covers:
1.  **Data Collection & Preprocessing**: Setting up the dataset and `tf.data` pipeline.
2.  **Model Building**: Using VGG19 for transfer learning.
3.  **Training**: Training and saving the model with optimizations.
4.  **Application Building**: Creating a Flask web app to serve the model (runnable in Colab via ngrok).

## 1. Prerequisites & Setup

Install necessary libraries and import dependencies.

In [ ]:
!pip install flask pyngrok tensorflow numpy matplotlib pandas

import tensorflow as tf
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model, load_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
from glob import glob

print(f"TensorFlow Version: {tf.__version__}")

# Check for GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus}")
    try:
        # Enable Mixed Precision for faster training on GPU
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("Mixed precision enabled")
    except Exception as e:
        print(f"Error enabling mixed precision: {e}")
else:
    print("No GPU detected. Running on CPU (this will be slow).")

## 2. Data Collection & Preprocessing

**Instructions:**
1.  **Mount Drive**: Run the cell below to connect your Google Drive.
2.  **Structure**: Ensure your data is in `MyDrive/dog_breed_identification/data/`.
3.  **Automatic Restructuring**: If your images are all in one folder, this script will try to use `labels.csv` to organize them into breed subfolders.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab, skipping Drive mount.")

# Define paths
base_dir = '/content/drive/MyDrive/dog_breed_identification'
data_dir = os.path.join(base_dir, 'data')
TRAIN_PATH = os.path.join(data_dir, 'train')
TEST_PATH = os.path.join(data_dir, 'test')
LABELS_CSV = os.path.join(base_dir, 'labels.csv')

# --- DATASET RESTRUCTURING LOGIC ---
def organize_dataset(train_path, labels_path):
    if not os.path.exists(labels_path):
        # Check if labels are in data dir instead
        labels_path = os.path.join(data_dir, 'labels.csv')
        if not os.path.exists(labels_path):
            print(f"WARNING: 'labels.csv' not found. Cannot organize dataset automatically.")
            return

    print(f"Found labels.csv at {labels_path}. Checking if organization is needed...")
    df = pd.read_csv(labels_path)
    
    # Check if images are already in folders or just flat files
    files_in_train = os.listdir(train_path)
    jpg_files = [f for f in files_in_train if f.lower().endswith('.jpg')]
    
    if len(jpg_files) > 0:
        print(f"Found {len(jpg_files)} images in root of train folder. Moving them to class subfolders...")
        # Create class folders
        breeds = df['breed'].unique()
        for breed in breeds:
            os.makedirs(os.path.join(train_path, breed), exist_ok=True)
            
        # Move files
        params = df.set_index('id')['breed'].to_dict()
        moved_count = 0
        for img_file in jpg_files:
            # File format usually ID.jpg
            img_id = os.path.splitext(img_file)[0]
            if img_id in params:
                breed = params[img_id]
                src = os.path.join(train_path, img_file)
                dst = os.path.join(train_path, breed, img_file)
                try:
                    shutil.move(src, dst)
                    moved_count += 1
                except Exception as e:
                    print(f"Error moving {img_file}: {e}")
        print(f"Successfully organized {moved_count} images into breed folders.")
    else:
        print("No flat images found in train folder. Assuming already organized.")

# Run organization if train path exists
if os.path.exists(TRAIN_PATH):
    organize_dataset(TRAIN_PATH, LABELS_CSV)

# --- IMAGE CONFIGURATION ---
IMAGE_SIZE = [224, 224]
BATCH_SIZE = 32

# Load Data using tf.data for better performance
if os.path.exists(TRAIN_PATH):
    print("Loading Training Set...")
    training_set = tf.keras.utils.image_dataset_from_directory(
        TRAIN_PATH,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=tuple(IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    print("Loading Validation Set...")
    validation_set = tf.keras.utils.image_dataset_from_directory(
        TRAIN_PATH,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=tuple(IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    class_names = training_set.class_names
    NUM_CLASSES = len(class_names)
    print(f"Detected {NUM_CLASSES} classes.")

    # Optimize datasets for performance
    AUTOTUNE = tf.data.AUTOTUNE
    
    def preprocess(image, label):
        image = tf.cast(image, tf.float32) / 255.0  # Rescale
        return image, label

    training_set = training_set.map(preprocess).cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    validation_set = validation_set.map(preprocess).cache().prefetch(buffer_size=AUTOTUNE)

else:
    print(f"ERROR: Dataset not found at {TRAIN_PATH}.")
    NUM_CLASSES = 0

## 3. Model Building (VGG19)

We use the VGG19 model pre-trained on ImageNet. We freeze the base layers and add our own fully connected layers.

In [ ]:
# Import VGG19
vgg = VGG19(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)

# Freeze existing weights
for layer in vgg.layers:
    layer.trainable = False

# Add Custom Layers
x = Flatten()(vgg.output)
prediction = Dense(NUM_CLASSES, activation='softmax')(x)

# Create Model
model = Model(inputs=vgg.input, outputs=prediction)
model.summary()

# Compile Model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## 4. Training the Model

Added `ModelCheckpoint` and `EarlyStopping` to prevent overfitting and save the best model.

In [ ]:
EPOCHS = 10 

if 'training_set' in locals() and NUM_CLASSES > 0:
    # Use ModelCheckpoint and EarlyStopping
    from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

    checkpoint = ModelCheckpoint('dog_breed_vgg19.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        training_set,
        validation_data=validation_set,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stop]
    )
    
    # Plot Results
    plt.plot(history.history['loss'], label='train loss')
    plt.plot(history.history['val_loss'], label='val loss')
    plt.legend()
    plt.show()
    
    plt.plot(history.history['accuracy'], label='train acc')
    plt.plot(history.history['val_accuracy'], label='val acc')
    plt.legend()
    plt.show()
else:
    print("Skipping training because no classes were found.")

## 5. Application Building (Flask)

We will now create a web application to serve our model.

### Step 5.1: Create Project Structure
We need a `templates` folder for HTML and an `uploads` folder for images.

In [ ]:
import os
os.makedirs('templates', exist_ok=True)
os.makedirs('uploads', exist_ok=True)

### Step 5.2: Create HTML Template
This creates `templates/index.html`.

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dog Breed Predictor</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body { background-color: #f8f9fa; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
        .container { max-width: 600px; margin-top: 50px; background: white; padding: 30px; border-radius: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        h1 { color: #2c3e50; margin-bottom: 30px; }
        .preview-image { max-width: 100%; height: auto; border-radius: 10px; margin-top: 20px; display: none; }
        #result { margin-top: 20px; font-size: 1.2rem; font-weight: bold; color: #27ae60; }
    </style>
</head>
<body>
    <div class="container text-center">
        <h1>🐶 Dog Breed Identifier</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <div class="mb-3">
                <label for="file" class="form-label">Upload a Dog Image</label>
                <input class="form-control" type="file" name="file" id="file" required onchange="previewimage(event)">
            </div>
            <button type="submit" class="btn btn-primary w-100">Predict Breed</button>
        </form>
        <img id="preview" class="preview-image" />
        {% if prediction %}
            <div id="result" class="alert alert-success mt-3">
                Prediction: {{ prediction }}
            </div>
        {% endif %}
    </div>

    <script>
        function previewimage(event) {
            var reader = new FileReader();
            reader.onload = function(){
                var output = document.getElementById('preview');
                output.src = reader.result;
                output.style.display = 'block';
            }
            reader.readAsDataURL(event.target.files[0]);
        }
    </script>
</body>
</html>

### Step 5.3: Create Flask Application

This creates `app.py`. Note: We generate the class dictionary first to safely embed it.

In [ ]:
# Prepare class indices dictionary
if 'class_names' in locals():
    # Convert list to dict: {0: 'breed1', 1: 'breed2'...}
    indices_dict = {i: name for i, name in enumerate(class_names)}
else:
    # Fallback
    indices_dict = {0: 'Unknown_Class'}

app_content = f"""
from flask import Flask, request, render_template
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import os
import sys
import tensorflow as tf

app = Flask(__name__)
MODEL_PATH = 'dog_breed_vgg19.h5'

# Ensure no GPU memory issues during inference if running in same process
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
except:
    pass

if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model file '{{MODEL_PATH}}' not found. Did you run the training cells?")
    # We continue so you can see the UI, but prediction will fail

model = None
try:
    if os.path.exists(MODEL_PATH):
        model = load_model(MODEL_PATH)
        print("Model loaded successfully.")
    else:
        print("Model not found, running without model... (prediction will fail)")
except Exception as e:
    print(f"ERROR loading model: {{e}}")

# Class Indices injected dynamically
CLASS_INDICES = {indices_dict}

def model_predict(img_path, model):
    if model is None:
        return "Model not loaded! Check server logs."
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        # Preprocess: Rescale to 0-1 as per training
        x = x / 255.0 
        x = np.expand_dims(x, axis=0)
        
        preds = model.predict(x)
        pred_class = np.argmax(preds, axis=1)[0]
        confidence = float(np.max(preds))
        
        result = CLASS_INDICES.get(pred_class, "Unknown")
        return f"{{result}} ({{confidence:.2%}})"
    except Exception as e:
        return f"Error: {{str(e)}}"

@app.route('/', methods=['GET'])
def index():
    return render_template('index.html', prediction=None)

@app.route('/predict', methods=['POST'])
def upload():
    if 'file' not in request.files:
        return "No file part"
    file = request.files['file']
    if file.filename == '':
        return "No selected file"
    
    filepath = os.path.join('uploads', file.filename)
    file.save(filepath)
    
    # Predict
    result = model_predict(filepath, model)
    return render_template('index.html', prediction=result)

if __name__ == '__main__':
    # Run on 0.0.0.0 to ensure it is accessible
    app.run(host='0.0.0.0', port=5000)
"""

with open('app.py', 'w') as f:
    f.write(app_content)

### Step 5.4: Run the App (Colab Compatible)

We use `pyngrok` to expose the local Flask server to the internet so you can access it from Colab.

**Important:** You need a free account at [dashboard.ngrok.com](https://dashboard.ngrok.com/signup) to get an authtoken.

In [ ]:
from pyngrok import ngrok
import os
import threading
import subprocess
import getpass
import time

# Terminate open tunnels if any
ngrok.kill()

print("To run the app, you need an ngrok authtoken.")
print("1. Sign up for free at https://dashboard.ngrok.com/signup")
print("2. Get your token at https://dashboard.ngrok.com/get-started/your-authtoken")
auth_token = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(auth_token)

# Open an HTTPs tunnel on port 5000
try:
    # FORCE IPv4 localhost to avoid ::1 connection refused errors
    public_url = ngrok.connect(addr="127.0.0.1:5000")
    print(f"\n * Public URL: {public_url}")
    print(" * Waiting for the Flask app to start...")

    # Run Flask in a subprocess to see errors
    process = subprocess.Popen(["python", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    # Give it a moment to possibly fail
    time.sleep(3)
    if process.poll() is not None:
        print("\nERROR: Flask app failed to start! Here is the error:")
        print(process.stderr.read())
    else:
        print(" * Flask app seems running. Click the Public URL above!")
        # Optionally print stdout/stderr lines in a non-blocking way if needed

except Exception as e:
    print(f"Error connecting ngrok: {e}")